# 3. Export Llama 3.2 1B to ExecuTorch (the file the app uses)
Needs a Hugging Face token with access to meta-llama/Llama-3.2-1B-Instruct. Output: `llama3_2_1b_v110.pte` (about 800 MB), made with executorch 1.1.0 to match the Android app. Copy it and `tokenizer.model` to `/data/local/tmp/llama/` on the phone.

Run the install cell, restart the session, then run from the next cell.

In [ ]:
!pip install -q "torch==2.10.0" "torchao==0.16.0" "executorch==1.1.0" huggingface_hub pytorch-tokenizers --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
!pip list 2>/dev/null | grep -i -E "^(executorch|torch|torchao) "

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import check. In the earlier run this import fixed "module torch.ao.quantization has no attribute quantizer".
import torch.ao.quantization.quantizer
from executorch.extension.llm.export.builder import LLMEdgeManager
print("imports ok")

In [ ]:
from huggingface_hub import login, snapshot_download
login()   # paste your Hugging Face token in the box that appears
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"],
                  local_dir="/content/waste-assistant/llama-1b")

In [ ]:
%%writefile /content/waste-assistant/export_config.yaml
base:
  model_class: llama3_2
  checkpoint: /content/waste-assistant/llama-1b/original/consolidated.00.pth
  params: /content/waste-assistant/llama-1b/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True

In [ ]:
%%bash
cd /content/waste-assistant
python -m executorch.extension.llm.export.export_llm --config export_config.yaml \
    +export.output_name="llama3_2_1b_v110.pte" \
    +export.max_seq_length=1024 +export.max_context_length=1024
ls -lh *.pte

In [ ]:
%%bash
DEST="/content/drive/MyDrive/Waste assistant"
mkdir -p "$DEST"
cp /content/waste-assistant/llama3_2_1b_v110.pte "$DEST/"
cp /content/waste-assistant/llama-1b/original/tokenizer.model "$DEST/"
ls -lh "$DEST"